Zadanie 13

In [33]:
import numpy as np
import pandas as pd

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.preprocessing import StandardScaler

np.random.seed(42)

data = fetch_california_housing()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name="MedHouseVal")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

base_model = LinearRegression()
base_model.fit(X_train, y_train)
r2_before = r2_score(y_test, base_model.predict(X_test))
print(f"R² przed feature engineering: {r2_before:.4f}")


# ===== FEATURE ENGINEERING  =====
X_train_fe = X_train.copy()
X_test_fe = X_test.copy()

# 1) interakcje
X_train_fe["MedInc_AveRooms"] = X_train_fe["MedInc"] * X_train_fe["AveRooms"]
X_test_fe["MedInc_AveRooms"] = X_test_fe["MedInc"] * X_test_fe["AveRooms"]

X_train_fe["MedInc_HouseAge"] = X_train_fe["MedInc"] * X_train_fe["HouseAge"]
X_test_fe["MedInc_HouseAge"] = X_test_fe["MedInc"] * X_test_fe["HouseAge"]

X_train_fe["Lat_Lon"] = X_train_fe["Latitude"] * X_train_fe["Longitude"]
X_test_fe["Lat_Lon"] = X_test_fe["Latitude"] * X_test_fe["Longitude"]

# 2) transformacje
X_train_fe["Log_Population"] = np.log1p(X_train_fe["Population"].clip(lower=0))
X_test_fe["Log_Population"] = np.log1p(X_test_fe["Population"].clip(lower=0))

X_train_fe["Log_AveOccup"] = np.log1p(X_train_fe["AveOccup"].clip(lower=0))
X_test_fe["Log_AveOccup"] = np.log1p(X_test_fe["AveOccup"].clip(lower=0))

# ===== model po FE =====
model = LinearRegression()
model.fit(X_train_fe, y_train)
r2_after = r2_score(y_test, model.predict(X_test_fe))
print(f"R² po feature engineering: {r2_after:.4f}")

improvement = (r2_after - r2_before) / abs(r2_before) * 100
print(f"Poprawa R²: {improvement:.2f}%")

print("\nNowe cechy dodane:")
print([
    "Interakcje: MedInc_AveRooms, MedInc_HouseAge, Lat_Lon",
    "Transformacje: Log_Population, Log_AveOccup",
])

# ===== Co poprawia najbardziej? =====
def score_variant(use_inter, use_trans):
    Xt = X_train.copy()
    Xv = X_test.copy()

    if use_inter:
        Xt["MedInc_AveRooms"] = Xt["MedInc"] * Xt["AveRooms"]
        Xv["MedInc_AveRooms"] = Xv["MedInc"] * Xv["AveRooms"]
        Xt["MedInc_HouseAge"] = Xt["MedInc"] * Xt["HouseAge"]
        Xv["MedInc_HouseAge"] = Xv["MedInc"] * Xv["HouseAge"]
        Xt["Lat_Lon"] = Xt["Latitude"] * Xt["Longitude"]
        Xv["Lat_Lon"] = Xv["Latitude"] * Xv["Longitude"]

    if use_trans:
        Xt["Log_Population"] = np.log1p(Xt["Population"].clip(lower=0))
        Xv["Log_Population"] = np.log1p(Xv["Population"].clip(lower=0))
        Xt["Log_AveOccup"] = np.log1p(Xt["AveOccup"].clip(lower=0))
        Xv["Log_AveOccup"] = np.log1p(Xv["AveOccup"].clip(lower=0))

    m = LinearRegression()
    m.fit(Xt, y_train)
    return r2_score(y_test, m.predict(Xv))

variants = {
    "bazowe": (False, False),
    "interakcje": (True, False),
    "transformacje": (False, True),
    "wszystko": (True, True)
}

print("\nWarianty (R² na teście):")
scores = []
for name, cfg in variants.items():
    r2 = score_variant(*cfg)
    scores.append((name, r2))
    print(f"{name:18s} -> {r2:.4f}, poprawa: {(r2 - r2_before) / abs(r2_before) * 100:.2f}%")

best = max(scores, key=lambda x: x[1])
print(f"\nNajlepszy wariant: {best[0]} (R²={best[1]:.4f}) poprawa: {(best[1] - r2_before) / abs(r2_before) * 100:.2f}%")

R² przed feature engineering: 0.5758
R² po feature engineering: 0.6338
Poprawa R²: 10.08%

Nowe cechy dodane:
['Interakcje: MedInc_AveRooms, MedInc_HouseAge, Lat_Lon', 'Transformacje: Log_Population, Log_AveOccup']

Warianty (R² na teście):
bazowe             -> 0.5758, poprawa: 0.00%
interakcje         -> 0.5803, poprawa: 0.79%
transformacje      -> 0.6330, poprawa: 9.94%
wszystko           -> 0.6338, poprawa: 10.08%

Najlepszy wariant: wszystko (R²=0.6338) poprawa: 10.08%
